# CrewAI

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **CrewAI** — a high-level framework for orchestrating **role-playing autonomous agents** that collaborate as a *crew* to finish a job. You describe each agent in plain language (a **role**, a **goal**, a **backstory**), hand the crew a list of **tasks**, pick a **process** (sequential or hierarchical), and CrewAI runs the show: it prompts each agent, passes one task's output to the next, and returns the final result.

Where [[langgraph]] gives you low-level graph control and [[autogen]] centers on free-form agent *conversations*, CrewAI's bet is **structure**: think in terms of a team with clear roles and a task list, and let the framework handle the wiring.

## 1. What & Why

**CrewAI** is an opinionated multi-agent framework built around a workplace metaphor: a **Crew** of **Agents**, each with a job description, working through a list of **Tasks**. You stay at the level of *"who does what, in what order"* and CrewAI compiles that into the prompts, tool calls, and hand-offs.

**The problem it solves.** A single LLM call struggles with jobs that have distinct *phases* — research, then analysis, then writing — because one prompt has to hold every instruction at once and the model context blurs the roles. The agentic answer is to split the work across specialists, but wiring that by hand (prompt templating, threading one step's output into the next, deciding who runs when) is repetitive boilerplate. CrewAI standardizes that pattern: declare specialists and a pipeline of tasks, and it handles delegation, context passing, and execution order.

**Reach for it when:** the work decomposes cleanly into roles (a researcher + a writer + an editor), you want to stand up a multi-agent pipeline fast with minimal code, and the control flow is mostly linear or a manager-delegates-to-workers shape. **Skip it when** you need fine-grained, cyclic, stateful control over the loop (use [[langgraph]]), or the whole job is a single tool-using agent (a plain ReAct loop is simpler), or you need deterministic, auditable branching — CrewAI's autonomy is a feature, not always a fit.

Two ways to build: **Crews** (autonomous agents that self-organize around tasks — the classic API) and **Flows** (newer, event-driven, code-first orchestration with explicit `@start`/`@listen` steps for when you want tighter control). They compose: a Flow step can kick off a Crew.

## 2. Mental Model

Think of CrewAI as **running a small project team.**

- An **Agent** is an employee with a job title (**role**), a mandate (**goal**), and a résumé (**backstory** that shapes its persona). It may carry **tools** and an **LLM**.
- A **Task** is a work ticket: a **description** of what to do and an **expected_output**. Each task is *assigned* to an agent.
- A **Crew** is the team plus a **process** that decides the running order.
- The **process** is the org chart: **sequential** = an assembly line (task 1 → task 2 → …, each task's output becomes context for the next); **hierarchical** = a **manager** agent that plans, delegates to workers, and reviews their results.

```
            ┌──────────────────────── Crew ────────────────────────┐
            │  process = sequential                                 │
            │                                                       │
   Task A ─▶ │  Agent: Researcher   ──output──▶ context for Task B   │
            │                                      │                │
   Task B ─▶ │  Agent: Writer        ◀────────────┘                 │
            │       │                                               │
            │       └──output──▶ FINAL RESULT                        │
            └───────────────────────────────────────────────────────┘

   sequential : A ──▶ B ──▶ C        (assembly line; output flows forward)
   hierarchical: Manager plans ──▶ delegates A,B,C to workers ──▶ reviews
```

The key intuition: **you write the job descriptions in natural language; CrewAI turns them into prompts and chains the outputs.** The agents are still LLMs under the hood — the framework is the org chart and the conveyor belt, not the intelligence.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`Agent`** | A role-playing worker. Core fields: `role`, `goal`, `backstory` (these become the system prompt), plus optional `tools`, `llm`, `allow_delegation`, `verbose`, `max_iter`. |
| **`Task`** | A unit of work: `description` (what to do), `expected_output` (what "done" looks like), and the `agent` it's assigned to. Optional `context` (other tasks whose output feeds in), `tools`, and `output_pydantic`/`output_json` for structured results. |
| **`Crew`** | The container: a list of `agents`, a list of `tasks`, and a `process`. You run it with `crew.kickoff(inputs={...})`; `inputs` interpolate `{placeholders}` in task/agent strings. |
| **`Process`** | Execution strategy. `Process.sequential` runs tasks in order, each output appended to the next task's context. `Process.hierarchical` adds a `manager_llm`/`manager_agent` that plans and delegates. |
| **Tools** | Capabilities an agent can call (search, file read, code exec, custom Python). From `crewai_tools` or defined with the `@tool` decorator / `BaseTool`. |
| **Context passing** | In a sequential crew, a task automatically receives prior task outputs as context; you can also pin specific upstream tasks via `Task(context=[...])`. |
| **`CrewOutput`** | What `kickoff()` returns: `.raw` (final text), `.tasks_output` (per-task results), `.token_usage`. Cast with `str()` for the final answer. |
| **Flows** | The code-first alternative: a `Flow` subclass with `@start()` and `@listen(...)` methods, explicit state, and event-driven routing. Use when you need control Crews abstract away. |

## 4. Setup

CrewAI is a single package; `crewai-tools` adds the prebuilt tool library. By default CrewAI talks to LLMs via [LiteLLM](https://docs.litellm.ai/), so it expects a provider key (e.g. `OPENAI_API_KEY`) — there is no offline default model.

```bash
pip install crewai            # core: Agent / Task / Crew / Flow
pip install 'crewai[tools]'   # + crewai_tools (search, scrape, RAG, file I/O, …)
export OPENAI_API_KEY=...     # or configure another LiteLLM provider
```

Requires **Python 3.10–3.13**. CrewAI is intentionally lean — it does **not** depend on LangChain.

Examples 1 & 2 below are **dependency-free**: they reimplement CrewAI's core (`Agent`, `Task`, `Crew`, sequential context passing) in ~50 lines of plain Python with a deterministic fake LLM, so the notebook runs anywhere with no install or API key. Example 3 shows the **real CrewAI code** and executes it only if `crewai` is installed *and* `OPENAI_API_KEY` is set; otherwise it prints the exact code you'd write.

In [1]:
# Installs are optional — examples 1 & 2 are pure stdlib and run offline.
# Uncomment to get the real library used in example 3:
# %pip install crewai

import importlib.util, os

has_crewai = importlib.util.find_spec("crewai") is not None
has_key    = bool(os.getenv("OPENAI_API_KEY"))
print("crewai installed:    ", has_crewai)
print("OPENAI_API_KEY present:", has_key)

crewai installed:     False
OPENAI_API_KEY present: False


## 5. Worked Examples

### Example 1 — `Agent` / `Task` / `Crew` from scratch (the sequential process)

CrewAI's core loop is small: each **agent** turns its role/goal/backstory plus the task into a prompt, an LLM answers, and in a **sequential** crew each task's output becomes **context** for the next task. Below we reimplement that core in plain Python — with a deterministic *fake* LLM standing in for a real model — so you can see exactly what `Crew(...).kickoff()` does under the hood. This mirrors `crewai`'s `Agent`, `Task`, and `Crew`.

In [2]:
from dataclasses import dataclass, field

# --- a deterministic stand-in for the LLM (no API key, fully offline) ---
def fake_llm(prompt: str) -> str:
    """Pretend-model: echoes a role-appropriate line so the pipeline is visible."""
    if "Researcher" in prompt:
        return "3 facts: (1) market up 12%, (2) churn down, (3) new competitor X."
    if "Writer" in prompt:
        # The writer can SEE the researcher's findings via the context below.
        ctx = prompt.split("CONTEXT:")[-1].strip()
        return f"Draft summary based on research -> {ctx[:80]}..."
    return "ok."


@dataclass
class Agent:
    role: str
    goal: str
    backstory: str
    llm: callable = fake_llm

    def system_prompt(self) -> str:
        return (f"You are a {self.role}. Goal: {self.goal}. "
                f"Backstory: {self.backstory}")


@dataclass
class Task:
    description: str
    expected_output: str
    agent: Agent
    context: list = field(default_factory=list)  # upstream task outputs


class Crew:
    """Minimal stand-in for crewai.Crew with Process.sequential."""
    def __init__(self, agents, tasks, process="sequential"):
        self.agents, self.tasks, self.process = agents, tasks, process

    def kickoff(self, inputs=None):
        inputs = inputs or {}
        outputs = []
        for task in self.tasks:
            # 1) interpolate {placeholders}, like inputs= in real CrewAI
            desc = task.description.format(**inputs)
            # 2) sequential process: prior task outputs feed in as CONTEXT
            ctx = "\n".join(task.context)
            prompt = (f"{task.agent.system_prompt()}\n"
                      f"TASK: {desc}\nEXPECTED: {task.expected_output}\n"
                      f"CONTEXT: {ctx}")
            result = task.agent.llm(prompt)         # the LLM call
            outputs.append(result)
            # 3) wire this output into every later task's context
            for later in self.tasks[self.tasks.index(task) + 1:]:
                later.context.append(result)
        return outputs[-1]                          # CrewOutput.raw


researcher = Agent(role="Researcher", goal="find {topic} facts",
                   backstory="A meticulous analyst.")
writer     = Agent(role="Writer", goal="summarize for execs",
                   backstory="A crisp business writer.")

t1 = Task("Research the latest on {topic}.", "3 bullet facts", researcher)
t2 = Task("Write an exec summary of {topic}.", "1 short paragraph", writer)

crew = Crew(agents=[researcher, writer], tasks=[t1, t2])
print(crew.kickoff(inputs={"topic": "Q3 metrics"}))

Draft summary based on research -> 3 facts: (1) market up 12%, (2) churn down, (3) new competitor X....


### Example 2 — Watching context flow through a 3-agent pipeline

The whole point of a sequential crew is the **conveyor belt**: researcher → writer → editor, where each step sees everything produced before it. Here we trace that flow explicitly so the hand-offs are visible — this is exactly the behavior `Process.sequential` gives you, and the shape you'd build for a real research-and-report crew.

In [3]:
def role_aware_llm(prompt: str) -> str:
    """Fake model whose reply depends on the role AND the context it received."""
    role = prompt.split("You are a ")[1].split(".")[0]
    ctx  = prompt.split("CONTEXT:")[-1].strip()
    saw  = f" (saw {ctx.count(chr(10)) + 1 if ctx else 0} prior output(s))"
    return {
        "Researcher": "FINDINGS: solar costs fell 40% since 2020.",
        "Writer":     "ARTICLE: Solar is now the cheapest power." + saw,
        "Editor":     "FINAL (polished): Solar: cheapest power on Earth." + saw,
    }.get(role, "ok.")

agents = [
    Agent("Researcher", "gather facts on {topic}", "PhD energy analyst", role_aware_llm),
    Agent("Writer",     "draft an article",        "science journalist", role_aware_llm),
    Agent("Editor",     "polish the final copy",   "ruthless copy editor", role_aware_llm),
]
tasks = [
    Task("Research {topic}.",          "key findings", agents[0]),
    Task("Write an article on {topic}.","a draft",     agents[1]),
    Task("Edit the article.",          "final copy",   agents[2]),
]

crew = Crew(agents, tasks)
final = crew.kickoff(inputs={"topic": "solar energy"})

# show the per-task trail (like CrewOutput.tasks_output)
for task in tasks:
    print(f"[{task.agent.role:11}] context carried in: {len(task.context)} item(s)")
print("\nFINAL RESULT:", final)

[Researcher ] context carried in: 0 item(s)
[Writer     ] context carried in: 1 item(s)
[Editor     ] context carried in: 2 item(s)

FINAL RESULT: FINAL (polished): Solar: cheapest power on Earth. (saw 2 prior output(s))


### Example 3 — The real thing: `Agent`, `Task`, `Crew` with a live LLM

This is the production pattern with real CrewAI classes. It calls a model, so it runs **only** if `crewai` is installed *and* `OPENAI_API_KEY` is set; otherwise it prints the exact code you'd write. Note the shape is identical to Examples 1–2 — agents with roles, tasks assigned to them, a sequential crew that flows context forward — just executed by the real framework against a live model.

In [4]:
SNIPPET = """
from crewai import Agent, Task, Crew, Process

researcher = Agent(
    role="Senior Research Analyst",
    goal="Find the most important recent facts about {topic}",
    backstory="You are a meticulous analyst who cites concrete numbers.",
    verbose=True,
    # llm="gpt-4o-mini"   # any LiteLLM model id; defaults via OPENAI_API_KEY
)
writer = Agent(
    role="Tech Content Writer",
    goal="Turn research into a crisp exec summary",
    backstory="You write tight, jargon-free briefings.",
    allow_delegation=False,
)

research = Task(
    description="Research the latest developments in {topic}.",
    expected_output="3-5 bullet points with concrete facts.",
    agent=researcher,
)
write = Task(
    description="Write a 1-paragraph executive summary of {topic}.",
    expected_output="One tight paragraph.",
    agent=writer,
    context=[research],          # explicitly feed the research output in
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research, write],
    process=Process.sequential,  # assembly line: research -> write
    verbose=True,
)
result = crew.kickoff(inputs={"topic": "on-device LLMs"})
print(result.raw)               # CrewOutput: .raw, .tasks_output, .token_usage
"""

import os, importlib.util
if importlib.util.find_spec("crewai") and os.getenv("OPENAI_API_KEY"):
    print("Running real CrewAI crew...\n")
    exec(SNIPPET)
else:
    print("Skipping live run (need `pip install crewai` + OPENAI_API_KEY).")
    print("The code you would run:\n")
    print(SNIPPET)

Skipping live run (need `pip install crewai` + OPENAI_API_KEY).
The code you would run:


from crewai import Agent, Task, Crew, Process

researcher = Agent(
    role="Senior Research Analyst",
    goal="Find the most important recent facts about {topic}",
    backstory="You are a meticulous analyst who cites concrete numbers.",
    verbose=True,
    # llm="gpt-4o-mini"   # any LiteLLM model id; defaults via OPENAI_API_KEY
)
writer = Agent(
    role="Tech Content Writer",
    goal="Turn research into a crisp exec summary",
    backstory="You write tight, jargon-free briefings.",
    allow_delegation=False,
)

research = Task(
    description="Research the latest developments in {topic}.",
    expected_output="3-5 bullet points with concrete facts.",
    agent=researcher,
)
write = Task(
    description="Write a 1-paragraph executive summary of {topic}.",
    expected_output="One tight paragraph.",
    agent=writer,
    context=[research],          # explicitly feed the research output i

## 6. Gotchas & Pitfalls

- **No free/offline default model.** CrewAI routes through LiteLLM and assumes a provider key. With no `OPENAI_API_KEY` (or other configured provider) `kickoff()` fails at the first LLM call. Set a key, or point `llm=` at a local model (e.g. an Ollama endpoint) before expecting it to run.
- **Vague tasks → vague (or runaway) agents.** The `description` and especially `expected_output` *are* the spec. Thin tasks produce meandering output, repeated tool calls, and burned tokens. Be concrete about the deliverable; that's what bounds the agent.
- **`allow_delegation=True` can loop.** Agents that delegate to each other (or a hierarchical manager that re-delegates) can ping-pong and inflate cost. Cap it with `max_iter`, set `allow_delegation=False` where you don't need it, and watch token usage.
- **Sequential ≠ smart routing.** `Process.sequential` is a fixed assembly line — it cannot branch, loop, or revisit a task based on a result. If you need conditional or cyclic control flow, use **Flows** or drop to [[langgraph]]; don't fake it with cleverly worded tasks.
- **Placeholders only fill from `inputs`.** `{topic}` in a description is interpolated from `crew.kickoff(inputs={...})`. A missing key raises a `KeyError`; a typo silently leaves the literal `{topic}` in the prompt. Keep names in sync.
- **Context is implicit but not infinite.** In a sequential crew every prior output is fed forward — great for small pipelines, but on long crews this bloats the prompt and cost. Use `Task(context=[...])` to pin *only* the relevant upstream tasks instead of dragging everything along.
- **Fast-moving API.** CrewAI iterates quickly; method names, `Process` options, and the tools package have shifted across versions. Pin a version and check the docs for the release you installed rather than trusting old blog posts.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs CrewAI |
|---|---|---|
| **CrewAI** | Role-based multi-agent **pipelines** you can describe in plain language; fast to stand up; sequential or manager-delegates shapes | Opinionated and higher-level — less control over the exact loop/state; autonomy can wander; not built for fine-grained cyclic control |
| **[[langgraph]]** | Stateful, **cyclic** agent loops; custom routing/branching; persistence + resume; human-in-the-loop | Lower-level — you wire nodes/edges/state yourself; more code for what CrewAI gives in a few declarations |
| **[[autogen]]** | Free-form **conversational** multi-agent systems (agents chatting to solve a task, code execution) | Conversation-centric rather than task/role-centric; less of a built-in "pipeline" abstraction |
| **[[langchain]] (LCEL)** | **Linear** dataflow: `retrieve → prompt → model → parse`; RAG chains | Not a multi-agent framework; no roles/delegation. Often a *component* inside an agent rather than the orchestrator |
| **A single ReAct agent** | One agent + a handful of tools; simple tool-use loops | No team/roles. If the job doesn't decompose into specialists, a crew is overkill |
| **CrewAI Flows** | Event-driven, code-first orchestration with explicit state and routing — *within* CrewAI | More control than Crews but more code; reach for it when the autonomous-crew abstraction is too loose |

**Rule of thumb:** if you can describe the job as *"a team of specialists working a checklist,"* CrewAI is the fastest path. If you find yourself fighting for conditional branches, loops, or precise state control, you've outgrown Crews — move to Flows or LangGraph.

## 8. Resources

- **Official docs** — https://docs.crewai.com/
- **Quickstart (build your first crew)** — https://docs.crewai.com/en/quickstart
- **Core concepts: Agents / Tasks / Crews** — https://docs.crewai.com/en/concepts/agents
- **Processes (sequential vs hierarchical)** — https://docs.crewai.com/en/concepts/processes
- **Flows (event-driven orchestration)** — https://docs.crewai.com/en/concepts/flows
- **Tools** — https://docs.crewai.com/en/concepts/tools
- **GitHub** — https://github.com/crewAIInc/crewAI